# Parse qualitative CG eval.

### Parse files into txt

In [2]:
import os
from docx import Document

# Path to your folder
folder_path = "../data/evaluation/STF_HC_Voto_Relatorio/resultados/Explanation_respostas_2"

# Loop through all files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith(".docx"):
        docx_path = os.path.join(folder_path, filename)
        txt_path = os.path.join(folder_path, os.path.splitext(filename)[0] + ".txt")

        # Read the docx file
        doc = Document(docx_path)
        text = "\n".join([para.text for para in doc.paragraphs])

        # Write to txt file
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(text)

        print(f"Converted: {filename} -> {os.path.basename(txt_path)}")

Converted: Explanation_Doc-1222_Hypernode_001.docx -> Explanation_Doc-1222_Hypernode_001.txt
Converted: Explanation_Doc-1222_Hypernode_026.docx -> Explanation_Doc-1222_Hypernode_026.txt
Converted: Explanation_Doc-162_Hypernode_029.docx -> Explanation_Doc-162_Hypernode_029.txt
Converted: Explanation_Doc-162_Hypernode_032.docx -> Explanation_Doc-162_Hypernode_032.txt
Converted: Explanation_Doc-171_Hypernode_000.docx -> Explanation_Doc-171_Hypernode_000.txt
Converted: Explanation_Doc-171_Hypernode_017.docx -> Explanation_Doc-171_Hypernode_017.txt
Converted: Explanation_Doc-1737_Hypernode_001.docx -> Explanation_Doc-1737_Hypernode_001.txt
Converted: Explanation_Doc-1737_Hypernode_035.docx -> Explanation_Doc-1737_Hypernode_035.txt
Converted: Explanation_Doc-1882_Hypernode_015.docx -> Explanation_Doc-1882_Hypernode_015.txt
Converted: Explanation_Doc-1882_Hypernode_018.docx -> Explanation_Doc-1882_Hypernode_018.txt
Converted: Explanation_Doc-292_Hypernode_019.docx -> Explanation_Doc-292_Hyper

In [2]:
import re
import pandas as pd
from pathlib import Path
from typing import List, Dict, Any
from sklearn.metrics import cohen_kappa_score
import numpy as np

# --- CONFIGURATION ---
# Update these paths to match your directory structure
REVIEWER_1_PATH = Path('../data/evaluation/STF_HC_Voto_Relatorio/resultados/Explanation_respostas_1/')
REVIEWER_2_PATH = Path('../data/evaluation/STF_HC_Voto_Relatorio/resultados/Explanation_respostas_2/')

METHOD_NAMES = {
    'm1': 'Method 1 (Semantic Search L0)',
    'm2': 'Method 2 (Semantic Search L1)',
    'm3': 'Method 3 (LLM-Based)',
    'm4': 'Method 4 (Most Representative)',
}


def identify_methods(candidates: List[Dict[str, Any]], word_group: List[str]) -> Dict[str, Dict[str, Any]]:
    """
    Identifies which candidate label corresponds to which method using heuristics.
    """
    identified_results = {}
    remaining_candidates = list(candidates)

    # Method 2 (Junk Method)
    junk_labels = ['pronunciado 2', 'aponta pronunciado foi']
    found_m2 = False
    for cand in list(remaining_candidates):
        if cand['label_text'].strip() in junk_labels:
            identified_results['m2'] = cand
            remaining_candidates.remove(cand)
            found_m2 = True
            break

    # Method 4 (Most Representative)
    found_m4 = False
    for cand in list(remaining_candidates):
        label_words = cand['label_text'].strip().split()
        if len(label_words) == 1 and label_words[0] in word_group:
            identified_results['m4'] = cand
            remaining_candidates.remove(cand)
            found_m4 = True
            break

    # Heuristic: Method 3 (LLM-Based) is a multi-word phrase not containing a comma
    found_m3 = False
    for cand in list(remaining_candidates):
        if len(cand['label_text'].strip().split()) > 1 and ',' not in cand['label_text']:
             identified_results['m3'] = cand
             remaining_candidates.remove(cand)
             found_m3 = True
             break

    # Heuristic: Method 1 (Semantic Search) often has a comma or is a single general word
    if len(remaining_candidates) > 0:
        for cand in list(remaining_candidates):
            if ',' in cand['label_text'] or len(cand['label_text'].strip().split()) == 1:
                if 'm1' not in identified_results:
                    identified_results['m1'] = cand
                    remaining_candidates.remove(cand)
                    break

    # Assign any remaining candidates to unfilled method slots
    if len(remaining_candidates) > 0:
        for cand in remaining_candidates:
            if 'm1' not in identified_results: identified_results['m1'] = cand
            elif 'm3' not in identified_results: identified_results['m3'] = cand
            elif 'm4' not in identified_results: identified_results['m4'] = cand
            elif 'm2' not in identified_results: identified_results['m2'] = cand

    return identified_results


def parse_survey_file(filepath: Path) -> Dict[str, Any]:
    """
    Parses a single survey file to extract all relevant information.
    """
    if not filepath.exists():
        return None
    content = filepath.read_text()

    doc_id_search = re.search(r"==== Doc Id: ====\s*([\d]+)", content)
    word_group_search = re.search(r"==== Abstract Node Word Group for evaluation: ====\s*(.+)", content)
    coherence_search = re.search(r"Q1.1:.*?Answer: \[(\d)\]", content, re.DOTALL)
    if not all([doc_id_search, word_group_search, coherence_search]):
        print("Skipping")
        return None # Skip malformed files

    doc_id = doc_id_search.group(1)
    word_group_str = word_group_search.group(1)
    word_group = [word.strip() for word in word_group_str.split(',')]
    coherence_score = int(coherence_search.group(1))

    if coherence_score < 3:
        print("Word group", word_group_str)

    candidate_sections = re.findall(r"## Candidate Label.*?Proposed Label: (.*?)\n(.*?)(?=## Candidate Label|--- PART 3)", content, re.DOTALL)
    candidates_data = []
    for label_text, section_content in candidate_sections:
        scores = re.findall(r"Answer: \[(\d)\]", section_content)
        if len(scores) == 3:
            candidates_data.append({
                'label_text': label_text.strip(),
                'faithfulness': int(scores[0]),
                'specificity': int(scores[1]),
                'plausibility': int(scores[2]),
            })

    ranking_section_search = re.search(r"--- PART 3:.*?Q3.1:(.*)", content, re.DOTALL)
    if not ranking_section_search:
        return None

    ranking_section = ranking_section_search.group(1)
    ranks = re.findall(r"Rank \[(\d)\] - (.*?)\n", ranking_section)
    rank_map = {label.strip(): int(rank) for rank, label in ranks}

    for cand in candidates_data:
        cand['rank'] = rank_map.get(cand['label_text'])

    identified_methods = identify_methods(candidates_data, word_group)

    result = {'doc_id': doc_id, 'word_group': word_group_str, 'coherence': coherence_score}
    for m_key, m_data in identified_methods.items():
        if m_data: # Ensure data was found
            result[f'{m_key}_faithfulness'] = m_data.get('faithfulness')
            result[f'{m_key}_specificity'] = m_data.get('specificity')
            result[f'{m_key}_plausibility'] = m_data.get('plausibility')
            result[f'{m_key}_rank'] = m_data.get('rank')

    return result


def main():
    """
    Main function to run the parsing, agreement calculation, and analysis.
    """
    # Find common survey files between the two reviewers
    try:
        r1_files = {f.name for f in REVIEWER_1_PATH.iterdir() if f.name.startswith('Explanation')}
        r2_files = {f.name for f in REVIEWER_2_PATH.iterdir() if f.name.startswith('Explanation')}
    except FileNotFoundError as e:
        print(f"Error: Directory not found. Please check your paths.")
        print(f"Details: {e}")
        return

    common_files = sorted(list(r1_files.intersection(r2_files)))

    if not common_files:
        print(f"Error: No matching survey files found between '{REVIEWER_1_PATH}' and '{REVIEWER_2_PATH}'.")
        return

    print(f"Found {len(common_files)} matching surveys to analyze.")

    all_results = []
    kappa_scores = []

    for filename in common_files:
        res1 = parse_survey_file(REVIEWER_1_PATH / filename)
        res2 = parse_survey_file(REVIEWER_2_PATH / filename)

        if res1 and res2:
            all_results.append(res1)
            all_results.append(res2)

            r1_ranks = [res1.get(f'm{i}_rank') for i in range(1, 5)]
            r2_ranks = [res2.get(f'm{i}_rank') for i in range(1, 5)]

            if None not in r1_ranks and None not in r2_ranks:
                kappa = cohen_kappa_score(r1_ranks, r2_ranks)
                kappa_scores.append(kappa)

    # --- Agreement ---
    avg_kappa = np.mean(kappa_scores) if kappa_scores else 0.0
    print("\n--- Inter-Annotator Agreement ---")
    print(f"Average Cohen's Kappa for comparative ranking: {avg_kappa:.3f}")

    if not all_results:
        print("\nNo valid data parsed. Cannot generate further results.")
        return

    df = pd.DataFrame(all_results).dropna()

    # --- Coherence ---
    print("\n--- Coherence of Generated Word Groups (Combined Reviewers) ---")
    total_surveys = len(df)
    coherent_surveys = df[df['coherence'] >= 4].shape[0]
    coherence_percentage = (coherent_surveys / total_surveys) * 100 if total_surveys > 0 else 0
    avg_coherence_score = df['coherence'].mean()

    print(">>> Overall (All Word Groups):")
    print(f"    Average coherence score (1-5 scale): {avg_coherence_score:.2f}")
    print(f"    Experts rated as coherent (score ≥ 4) in {coherent_surveys}/{total_surveys} cases ({coherence_percentage:.2f}%).")

    df_multi_word = df[df['word_group'].str.contains(',')]
    if not df_multi_word.empty:
        total_multi_word = len(df_multi_word)
        coherent_multi_word = df_multi_word[df_multi_word['coherence'] >= 4].shape[0]
        percentage_multi_word = (coherent_multi_word / total_multi_word) * 100 if total_multi_word > 0 else 0
        avg_coherence_multi_word = df_multi_word['coherence'].mean()
        print("\n>>> For Multi-Word Groups Only (excluding isolated words):")
        print(f"    Average coherence score (1-5 scale): {avg_coherence_multi_word:.2f}")
        print(f"    Experts rated as coherent (score ≥ 4) in {coherent_multi_word}/{total_multi_word} cases ({percentage_multi_word:.2f}%).")

    # --- Aggregated Metrics with Std Dev ---
    aggregated_data = []
    for m_key, m_name in METHOD_NAMES.items():
        data = { 'CG Method': m_name }
        for metric in ['rank', 'faithfulness', 'specificity', 'plausibility']:
            col_name = f'{m_key}_{metric}'
            data[f'Avg {metric}'] = df[col_name].mean()
            data[f'Std {metric}'] = df[col_name].std()
        aggregated_data.append(data)
    agg_df = pd.DataFrame(aggregated_data).set_index('CG Method')

    # --- Generate LaTeX Table ---
    best_rank = agg_df['Avg rank'].min()
    best_faith = agg_df['Avg faithfulness'].max()
    best_spec = agg_df['Avg specificity'].max()
    best_plaus = agg_df['Avg plausibility'].max()

    latex_string = """
\\begin{table}[h]
\\centering
\\caption{Aggregated results (avg $ \\pm $ std) from the expert evaluation survey.}
\\renewcommand{\\arraystretch}{1.2}
\\label{tab:expert_survey_combined_std}
\\begin{tabular}{@{}lcccc@{}}
\\toprule
\\textbf{CG Method} & \\textbf{Rank} & \\textbf{Faithfulness} & \\textbf{Specificity} & \\textbf{Plausibility} \\\\
\\midrule"""

    for index, row in agg_df.iterrows():
        # Format strings as 'avg ± std'
        rank_val = f"{row['Avg rank']:.1f} $\\pm$ {row['Std rank']:.1f}"
        faith_val = f"{row['Avg faithfulness']:.1f} $\\pm$ {row['Std faithfulness']:.1f}"
        spec_val = f"{row['Avg specificity']:.1f} $\\pm$ {row['Std specificity']:.1f}"
        plaus_val = f"{row['Avg plausibility']:.1f} $\\pm$ {row['Std plausibility']:.1f}"

        # Determine which full string to bold
        rank_str = f"\\textbf{{{rank_val}}}" if row['Avg rank'] == best_rank else rank_val
        faith_str = f"\\textbf{{{faith_val}}}" if row['Avg faithfulness'] == best_faith else faith_val
        spec_str = f"\\textbf{{{spec_val}}}" if row['Avg specificity'] == best_spec else spec_val
        plaus_str = f"\\textbf{{{plaus_val}}}" if row['Avg plausibility'] == best_plaus else plaus_val

        latex_string += f"\n{index} & {rank_str} & {faith_str} & {spec_str} & {plaus_str} \\\\"

    latex_string += """
\\bottomrule
\\end{tabular}
\\end{table}"""

    print("\n--- LaTeX Table for Qualitative Evaluation (Combined Reviewers) ---")
    print(latex_string)

if __name__ == '__main__':
    main()








Found 18 matching surveys to analyze.
Word group expedição, objetivando, desatendimento
Word group hospital
Word group hospital
Word group termos, exame
Word group atos
Word group atos
Word group emerson, voto, corréus
Word group s, deduzido, extinta
Word group s, relatório, para, objetivando
Word group aparelhos, oliveira, de, cristiano
Word group violados, efêmero, de
Word group mogi, de
Word group mogi, de
Word group apresentação, indicada, relatório
Word group rodrigues, oliveira
Word group senhor

--- Inter-Annotator Agreement ---
Average Cohen's Kappa for comparative ranking: 0.082

--- Coherence of Generated Word Groups (Combined Reviewers) ---
>>> Overall (All Word Groups):
    Average coherence score (1-5 scale): 2.82
    Experts rated as coherent (score ≥ 4) in 9/34 cases (26.47%).

>>> For Multi-Word Groups Only (excluding isolated words):
    Average coherence score (1-5 scale): 3.00
    Experts rated as coherent (score ≥ 4) in 9/28 cases (32.14%).

--- LaTeX Table for Qual